## Step 1: Load and Explore the Data
#### - Load movies.csv
#### - Clean and combine features for content-based modeling

In [58]:
import numpy as np 
import pandas as pd 

#loading dataset
movie = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')

#merging both datasets on title 
movies = movie.merge(credits, on='title')

# Keep important columns
movies = movies[['id', 'title', 'genres', 'overview', 'keywords', 'cast', 'crew', 'popularity']]
# Drop rows with null values
movies.dropna(inplace=True)

# dropna() leaves gaps in the row labels; the similarity matrix is indexed by position,
# so reset the index to keep labels and positions aligned
movies = movies.reset_index(drop=True)


C:\Users\acer\AppData\Local\Temp\ipykernel_5836\1649104019.py:6: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,25

In [59]:
movies.head()

,id,title,genres,overview,keywords,cast,crew,popularity
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","In the 22nd century, a paraplegic Marine is di...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",150.437577
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","Captain Barbossa, long believed to be dead, ha...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",139.082615
2,206647,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",A cryptic message from Bond’s past sends him o...,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",107.376788
3,49026,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",Following the death of District Attorney Harve...,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",112.312950
4,49529,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","John Carter is a war-weary, former military ca...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",43.926995


In [60]:
#breaking sentences into words in overview column 
def break_sent(x):
    return x.split()
movies['overview'] = movies['overview'].apply(break_sent)

In [61]:
#processing the genres column 
import ast 
def process_gen(gen):
    x = []
    for i in ast.literal_eval(gen): #converts the text into dictionary form 
        x.append(i['name']) #keep only the name 
    return x

movies['genres'] = movies['genres'].apply(process_gen)

In [62]:
#processing the keywords column 
def process_key(key):
    x = []
    for i in ast.literal_eval(key): 
        x.append(i['name']) 
    return x

movies['keywords'] = movies['keywords'].apply(process_key)

In [63]:
movies.head()

,id,title,genres,overview,keywords,cast,crew,popularity
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[In, the, 22nd, century,, a, paraplegic, Marin...","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",150.437577
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Captain, Barbossa,, long, believed, to, be, d...","[ocean, drug abuse, exotic island, east india ...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",139.082615
2,206647,Spectre,"[Action, Adventure, Crime]","[A, cryptic, message, from, Bond’s, past, send...","[spy, based on novel, secret agent, sequel, mi...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",107.376788
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[Following, the, death, of, District, Attorney...","[dc comics, crime fighter, terrorist, secret i...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",112.312950
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[John, Carter, is, a, war-weary,, former, mili...","[based on novel, mars, medallion, space travel...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",43.926995


In [64]:
#processing the cast column
def process_cast(cast):
    x = []
    count = 0
    for i in ast.literal_eval(cast): 
        if count <5: #getting the top 5 cast members
            x.append(i['name']) 
            count+=1
    return x

movies['cast'] = movies['cast'].apply(process_cast)

In [65]:
movies.head()

,id,title,genres,overview,keywords,cast,crew,popularity
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[In, the, 22nd, century,, a, paraplegic, Marin...","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",150.437577
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Captain, Barbossa,, long, believed, to, be, d...","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley, ...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",139.082615
2,206647,Spectre,"[Action, Adventure, Crime]","[A, cryptic, message, from, Bond’s, past, send...","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux, R...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",107.376788
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[Following, the, death, of, District, Attorney...","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman, A...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",112.312950
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[John, Carter, is, a, war-weary,, former, mili...","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton,...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",43.926995


In [66]:
#processing the crew column to get the director 
#handling errors because simple code like above wasn't working 
def process_crew(crew):
    if crew is None or (isinstance(crew, float) and pd.isna(crew)): #handling missing values, NaN is stored as float
        return []

    if isinstance(crew, list): #if crew is already a list 
        crew_list = crew
    else:
        try: #otherwise converting string to Python object
            crew_list = ast.literal_eval(crew) #converts to a Python list
        except:
            return [] #if it doesn't work, return an empty list

    return [i['name'] for i in crew_list if i['job'] == 'Director'] #extract the director name

movies['crew'] = movies['crew'].apply(process_crew)

In [67]:
movies.head()

,id,title,genres,overview,keywords,cast,crew,popularity
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[In, the, 22nd, century,, a, paraplegic, Marin...","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...",[James Cameron],150.437577
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Captain, Barbossa,, long, believed, to, be, d...","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley, ...",[Gore Verbinski],139.082615
2,206647,Spectre,"[Action, Adventure, Crime]","[A, cryptic, message, from, Bond’s, past, send...","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux, R...",[Sam Mendes],107.376788
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[Following, the, death, of, District, Attorney...","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman, A...",[Christopher Nolan],112.312950
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[John, Carter, is, a, war-weary,, former, mili...","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton,...",[Andrew Stanton],43.926995


## Step 2: Content-Based Filtering
#### - Create a tags column (genre, overview, cast, etc.)
#### - Vectorize using CountVectorizer or TF-IDF
#### - Compute cosine similarity

In [72]:
#creating a tag with important columns
movies['tags'] = (
    movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['crew']
)

#cleaning up spaces
movies['tags'] = movies['tags'].apply(lambda x: [i.replace(" ", "") for i in x])
#lambda arguments:expression, defines a small function (no need to define one)

movies = movies[['id', 'title', 'tags', 'popularity']]

In [74]:
#preprocessing tags before vectorisation 
import nltk 
from nltk.stem import WordNetLemmatizer #lemmatizer converts word to its base form (lemma)
#nltk.download('wordnet')
#nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()
def lemmatize_text(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text])
movies['tags'] = movies['tags'].apply(lemmatize_text)

#CountVectorizer 
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    max_features=500,
    stop_words='english'
)

vectors = cv.fit_transform(movies['tags']).toarray()

In [76]:
from sklearn.metrics.pairwise import cosine_similarity 

similarity = cosine_similarity(vectors)

## Step 3: Save the model

In [80]:
#saving the model 
import pickle 
pickle.dump(movies, open('model.pkl', 'wb')) #wb = write in bindary code 
pickle.dump(similarity, open('similarity.pkl', 'wb'))

## Step 4: Test Recommendations
#### Create a function to recommend movies based on the selected title.

In [ ]:
def get_recommendations(movie_title, top_n=5):
    movie_index = movies[movies['title'] == movie_title].index[0]   # row position (index was reset above)
    scores = list(enumerate(similarity[movie_index]))               # (movie_index, similarity) pairs
    ranked = sorted(scores, key=lambda x: x[1], reverse=True)       # most similar first
    return [movies.iloc[i].title for i, _ in ranked[1:top_n + 1]]   # skip the movie itself

get_recommendations('Avatar')